In [1]:
import arcpy
import os
import pandas as pd

# 1. 基础路径设置
tif_folder = r"d:\Project\邓旭婷\阳宗海\第二波\9年成果"
shp_file = r"d:\Project\邓旭婷\阳宗海\第二波\9年成果\湖体分区84.shp"
out_folder = r"d:\Project\邓旭婷\阳宗海\第二波\分区提取及统计" # 结果将统一保存在这个新文件夹

# 如果输出目录不存在则创建
if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# 波段与水质参数对应关系
band_mapping = {
    1: 'DO',
    2: 'COD',
    3: 'TP',
    4: 'TN',
    5: 'Chla'
}

# 环境设置与 Spatial 权限检查
arcpy.env.workspace = tif_folder
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

# 2. 读取糊体分区矢量
# 如果 shp 内有特定的字段用于分区命名（例如"Name"），可以把 "OID@" 改为 "Name"
zone_field = "name_1" 
zones = []
with arcpy.da.SearchCursor(shp_file, [zone_field, "SHAPE@"]) as cursor:
    for row in cursor:
        zones.append({'zone_id': row[0], 'geometry': row[1]})

print(f"成功读取到 {len(zones)} 个湖体分区。开始掩膜拆分并统计...")

# 3. 遍历 TIF 文件并处理
tif_files = [f for f in os.listdir(tif_folder) if f.endswith(".tif")]

# 定义空列表，用于汇总存储所有的统计结果
stats_list = []

for tif in tif_files:
    # 从文件名提取年份，例如 Yangzonghai_1986.tif 提取为 1986
    year = tif.split('_')[-1].replace('.tif', '')
    tif_path = os.path.join(tif_folder, tif)
    
    print(f"\n---> 正在处理 {year} 年的 TIF数据: {tif}")
    
    for zone in zones:
        z_id = zone['zone_id']
        geom = zone['geometry']
        zone_name = f"Zone_{z_id}"  # 如果前面换了 Name 字段，这里可以将 "Zone_" 前缀去掉
        
        # 拆分后的多波段 TIF 输出路径
        out_tif_name = f"{zone_name}_{year}.tif"
        out_tif_path = os.path.join(out_folder, out_tif_name)
        
        # 掩膜提取（按当前矢量面切割 TIF 并保存）
        try:
            extracted_raster = arcpy.sa.ExtractByMask(tif_path, geom)
            extracted_raster.save(out_tif_path)
        except Exception as e:
            print(f"  [警告] 提取 {year} 年 {zone_name} 分区失败（或者该分区无重叠）。错误详情: {e}")
            continue
            
        # 确保软件完整计算了新建结果的统计信息
        arcpy.management.CalculateStatistics(out_tif_path)
        
        print(f"  - 成功拆分并保存 {out_tif_name}，正在分波段统计水质参数...")
        
        # 遍历读取5个波段的统计值
        for band_idx, param in band_mapping.items():
            # TIF 文件的内部波段名默认命名为 Band_1, Band_2 ...
            band_path = os.path.join(out_tif_path, f"Band_{band_idx}")
            
            try:
                min_val = float(arcpy.management.GetRasterProperties(band_path, "MINIMUM").getOutput(0))
                max_val = float(arcpy.management.GetRasterProperties(band_path, "MAXIMUM").getOutput(0))
                mean_val = float(arcpy.management.GetRasterProperties(band_path, "MEAN").getOutput(0))
                std_val  = float(arcpy.management.GetRasterProperties(band_path, "STD").getOutput(0))
            except Exception as e:
                print(f"    [警告] 无法读取 {param} (Band_{band_idx}) 的统计值。")
                min_val, max_val, mean_val, std_val = None, None, None, None
            
            # 将该波段的汇总写入字典
            stats_list.append({
                '分区': zone_name,
                '水质参数': param,
                '年份': year,
                '最小值': min_val,
                '最大值': max_val,
                '平均值': mean_val,
                '标准差': std_val
            })

# 4. 利用 Pandas 将数据汇总输出为 Excel 数据表
print("\n数据处理完毕，开始导出最终的汇总表格...")

if stats_list:
    df = pd.DataFrame(stats_list)
    
    # 【业务需求】：最终得到每个分区下的每个水质参数的每个年份的数据表
    # 策略：以“分区”分组，每个分区生成 1 个 Excel文件；
    # 在这个 Excel 内部，以“水质参数”（DO、COD等）作Sheet区分表，Sheet内行为每年记录
    for zone_name, group_by_zone in df.groupby('分区'):
        excel_path = os.path.join(out_folder, f"{zone_name}_水质参数统计表.xlsx")
        
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            for param, group_by_param in group_by_zone.groupby('水质参数'):
                # 按照年份降序列排序排版
                param_df = group_by_param.sort_values(by='年份')
                
                # 写入到对应名称的 Sheet 标签页中，不要前置的行索引编号
                param_df.to_excel(writer, sheet_name=param, index=False)
                
        print(f"已生成【{zone_name}】的统计数据表 -> {excel_path}")

print(f"\n🎉 全部任务已成功完成！\n切分后的各区各年TIF文件 以及 Excel数据汇总报表 均已统一保存在：\n{out_folder}")


成功读取到 4 个湖体分区。开始掩膜拆分并统计...

---> 正在处理 1986 年的 TIF数据: Yangzonghai_1986.tif
  - 成功拆分并保存 Zone_湖体W_1986.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体N_1986.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体S_1986.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体E_1986.tif，正在分波段统计水质参数...

---> 正在处理 1990 年的 TIF数据: Yangzonghai_1990.tif
  - 成功拆分并保存 Zone_湖体W_1990.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体N_1990.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体S_1990.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体E_1990.tif，正在分波段统计水质参数...

---> 正在处理 1995 年的 TIF数据: Yangzonghai_1995.tif
  - 成功拆分并保存 Zone_湖体W_1995.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体N_1995.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体S_1995.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体E_1995.tif，正在分波段统计水质参数...

---> 正在处理 2000 年的 TIF数据: Yangzonghai_2000.tif
  - 成功拆分并保存 Zone_湖体W_2000.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体N_2000.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体S_2000.tif，正在分波段统计水质参数...
  - 成功拆分并保存 Zone_湖体E_2000.tif，正在分波段统计水质参数...

---> 正在处理 2005 年的 TIF数据: Yangzonghai_2005.tif
  - 成功拆分并保存 Zone_湖